<a href="https://colab.research.google.com/github/ornab74/blog-write-gemma/blob/main/qroadscan_blog_writer_gemma4_llama-cpp-python-working-logs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QRoadScan Gemma 4 E2B llama-cpp-python CUDA Hive Blog Forge — V8 Colab CMAKE_ARGS install

This notebook uses a **Colab GPU setup with `llama-cpp-python` built with CUDA using `CMAKE_ARGS="-DGGML_CUDA=on"` from a normal Colab Python cell**. It does **not** use Homebrew, LiteRT-LM, a standalone `llama.cpp` checkout, or `llama-server`.

What changed in V8:

- The first executable cell uses Colab `!pip` / `%env` commands, not `%%bash`.
- It sets `CMAKE_ARGS=-DGGML_CUDA=on` and `FORCE_CMAKE=1` before installing `llama-cpp-python`.
- It installs build helpers with pip: `cmake`, `ninja`, `scikit-build-core`, `setuptools`, and `wheel`.
- It downloads `ggml-org/gemma-4-E2B-it-GGUF` as a local GGUF file through `huggingface_hub`.
- It loads `gemma-4-e2b-it-Q8_0.gguf` with the Python `llama_cpp.Llama` API and GPU offload.
- It keeps the QRoadScan hive, RAG surfaces, citation ledger, intercom memory, quality audit, and ZIP download cells.

Run order:

1. In Colab, choose **Runtime → Change runtime type → GPU**.
2. Run **Cell 0**. It installs dependencies and builds/installs `llama-cpp-python` with CUDA enabled.
3. Run the remaining cells top to bottom.
4. Edit **Cell 1** for topic count, worker count, model settings, and brand facts.
5. Run the final download helper cell to download the generated ZIP.

Notes:

- The requested `CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python` setup compiles the Python package in Colab, so this install cell can take longer than using a prebuilt wheel.
- The model is shared as one local `Llama()` instance. The article hive can still schedule multiple workers, but model calls are protected by a lock so the shared context stays stable.
- Gemma 4 E2B GGUF can be memory-heavy. On small Colab GPUs, reduce `N_CTX`, `N_GPU_LAYERS`, `ARTICLE_COUNT`, or `GEMMA_THREAD_COUNT` in Cell 1 if the runtime runs out of memory.
- If Hugging Face access is rate-limited or the model repo requires terms acceptance, set `HF_TOKEN` in Colab secrets or run `huggingface-cli login` before the model download cell.


In [1]:
# Cell 0 — llama-cpp-python CUDA wheel install for Colab CUDA 13 driver
# No CMAKE_ARGS. No FORCE_CMAKE. No source build.

!nvidia-smi
!pip install pennylane
!pip uninstall -y llama-cpp-python llama_cpp_python
!pip install -q --upgrade pip setuptools wheel

# CUDA 13 driver can run CUDA 12.x wheels.
# cu124 is the safest prebuilt index to try in Colab.
!pip install --upgrade --prefer-binary --only-binary=:all: \
  llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import importlib.metadata as md
import llama_cpp

print("llama-cpp-python:", md.version("llama-cpp-python"))
print("GPU offload support:", llama_cpp.llama_supports_gpu_offload())

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError("llama-cpp-python installed, but GPU offload is not enabled.")

Thu Apr 30 00:20:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


llama-cpp-python: 0.3.21
GPU offload support: True


In [2]:
# Cell 1 — SINGLE CONTROL CELL: topics, hive size, brand facts, llama-cpp-python generation mode
# Edit only this cell for normal use.

from pathlib import Path
import os, textwrap

# Generate 1 to 42 Markdown posts in one run.
ARTICLE_COUNT = 10  # <-- set 1..42

# Run 1 to 10 editorial worker threads through one shared llama-cpp-python Llama context.
# On a T4, start with 2-4. Calls into the single model context are serialized for safety.
GEMMA_THREAD_COUNT = 4  # <-- set 1..10
AUTO_SCALE_THREADS_TO_T4 = True
T4_TARGET_VRAM_UTILIZATION = 0.92
APPROX_VRAM_GB_PER_WORKER = 0.85
MIN_FREE_VRAM_GB_AFTER_LOAD = 2.00
THREAD_STARTUP_STAGGER_SECONDS = 0.25

# Hive/intercom knobs.
HIVE_MODE = "mesh"  # solo, queue, mesh. mesh uses the live inter-worker surface.
INTERCOM_ROUNDS = 1  # 0..3 extra bridge notes per article. Higher = richer but slower.
INTERCOM_MAX_SURFACE_CHARS = 16000
INTERCOM_PACKET_CHARS = 2400
INTERCOM_MEMORY_ITEMS = 140
ENABLE_SUMMA_COMPACTION = True
RAG_TOP_K = 12

# Add as many topics as you want. If ARTICLE_COUNT exceeds this list, topic variants are auto-expanded.
BLOG_TOPICS = [
    "How QRoadScan turns traffic risk signals into calmer route decisions",
    "Why a live road risk colorwheel can help drivers understand hazards faster",
    "Road hazard alerts, confidence scoring, and the future of safer commutes",
    "AI-assisted traffic awareness without overwhelming the driver",
    "From route scans to saved reports: what a road safety dashboard should explain",
    "How predictive safety context can reduce uncertainty before a trip",
]

BRAND_NAME = "QRoadScan.com"
BRAND_URL = "https://qroadscan.com"
BRAND_VOICE = "human, practical, safety-aware, technical but readable, quietly futuristic"
TARGET_READER = "drivers, road-safety researchers, fleet operators, traffic-tech builders, and safety-conscious commuters"

BRAND_CONTEXT = """
QRoadScan.com is a road intelligence and safety awareness product focused on live traffic risk visualization, route scans, road hazard alerts, saved reports, and AI-assisted driving safety insights.
The product concept compresses changing route signals into a readable risk pulse: one reading, one confidence score, practical reasons, and driver-ready next steps.
Core motifs: live risk map, road hazard awareness, safer route planning, colorwheel feedback, route scan reports, confidence scoring, calmer decisions, dashboard intelligence, and real-world driving context.
""".strip()

# Local RAG surfaces: add factual notes, product docs, landing page copy, research summaries, or quotes you own.
# Each item can have title, url, and body. The writer cites these with local surface citations.
EXTRA_RAG_SURFACES = [
    {
        "title": "QRoadScan product positioning",
        "url": BRAND_URL,
        "body": BRAND_CONTEXT,
    },
    {
        "title": "QRoadScan blog direction",
        "url": f"{BRAND_URL}/blog",
        "body": "Short reads about road intelligence, safer routes, traffic risk, road hazards, and product updates.",
    },
]

# Optional folder for extra .txt, .md, .json, .csv files you upload into Colab.
# Example: upload docs to /content/qroadscan_sources before running the batch cell.
SOURCE_FOLDER = Path("/content/qroadscan_sources")
OUTPUT_DIR = Path("/content/qroadscan_blog_markdowns_v5_llamacpp_python")

# Agent depth: "fast" = planner + draft, "deep" = guard + editor, "hive-deep" = deep plus intercom bridge.
AGENT_DEPTH = "hive-deep"

# Style knobs.
WORDS_PER_ARTICLE = 1200
TEMPERATURE_STYLE = "balanced-human"  # options: crisp, balanced-human, cinematic, founder-note, field-notes
INCLUDE_TECHNICAL_APPENDIX = True
INCLUDE_MEDIUM_READY_FRONTMATTER = True
INCLUDE_LOCAL_CITATION_LEDGER = True

# Worker personas rotate across the hive so tiny models behave more like a coordinated editorial desk.
WORKER_PERSONAS = [
    "route-safety systems writer: concrete, calm, operational",
    "product narrative strategist: crisp framing, founder-level clarity",
    "driver empathy editor: practical scenes, low hype, human rhythm",
    "citation sentinel: cautious claims, surface-first evidence discipline",
    "dashboard UX explainer: turns telemetry into readable decisions",
    "future-of-mobility essayist: speculative only when clearly labeled",
    "fleet operations analyst: repeatable workflows and risk communication",
    "technical translator: plain-language systems explanations",
    "Medium polish editor: strong hooks, clean sections, no filler",
    "intercom synthesizer: finds patterns across the hive memory",
]

# llama-cpp-python / Gemma 4 knobs.
# Google’s Gemma llama.cpp guide uses this GGUF repo directly; here we download the GGUF and load it with the Python binding.
GEMMA_MODEL_REPO_ID = "ggml-org/gemma-4-E2B-it-GGUF"
GEMMA_GGUF_FILENAME = "gemma-4-e2b-it-Q8_0.gguf"
MODEL_FILE = GEMMA_MODEL_REPO_ID
EXPECTED_HASH = ""  # Optional SHA-256 for the downloaded GGUF; leave blank unless you want to pin a file.

# GPU offload. 999 means "try to offload all layers" in practice for small/medium GGUF models.
INFERENCE_BACKEND = "CUDA"
LLAMA_N_GPU_LAYERS = 999
LLAMA_CONTEXT_SIZE = 8192
LLAMA_BATCH_SIZE = 1024
LLAMA_UBATCH_SIZE = 256
LLAMA_FLASH_ATTN = True
LLAMA_VERBOSE = True
LLAMA_LOAD_TIMEOUT_SECONDS = 480

# Generation settings for llama-cpp-python chat completions.
MAX_PROMPT_CHARS = 24000
MAX_ARTICLE_CHARS = 32000
LLAMA_MAX_TOKENS = 1700
LLAMA_TEMPERATURE = 0.70
LLAMA_TOP_P = 0.95
LLAMA_TOP_K = 64
LLAMA_REPEAT_PENALTY = 1.05
ENABLE_ENGINE_WARMUP = False
WARMUP_PROMPT = "Return exactly: hive worker ready"

# Hive telemetry and local quality audit knobs.
ENABLE_HIVE_TELEMETRY = True
HIVE_TELEMETRY_INTERVAL_SECONDS = 4.0
ENABLE_LOCAL_QUALITY_AUDIT = True
QUALITY_MIN_CITATIONS = 1
QUALITY_WARN_PHRASES = [
    "in today's fast-paced world",
    "game changer",
    "revolutionary",
    "seamlessly",
    "unlock the power",
]

# Optional authenticated Hugging Face download support. Public models do not need this,
# but gated/private mirrors can use any one of these Colab Secret names.
HF_TOKEN_SECRET_NAMES = ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGING_FACE_HUB_TOKEN"]
EXPORT_COLAB_SECRETS_TO_ENV = True

print(f"Configured {ARTICLE_COUNT} article(s) for {BRAND_NAME}.")
print(f"Requested Gemma workers: {GEMMA_THREAD_COUNT} (auto-scale={AUTO_SCALE_THREADS_TO_T4})")
print(f"Model: {GEMMA_MODEL_REPO_ID}")
print(f"GGUF file: {GEMMA_GGUF_FILENAME}")
print(f"Output folder: {OUTPUT_DIR}")


Configured 10 article(s) for QRoadScan.com.
Requested Gemma workers: 4 (auto-scale=True)
Model: ggml-org/gemma-4-E2B-it-GGUF
GGUF file: gemma-4-e2b-it-Q8_0.gguf
Output folder: /content/qroadscan_blog_markdowns_v5_llamacpp_python


In [3]:
# Cell 2 — llama-cpp-python + Gemma 4 GGUF runtime prep
# Direct GGUF download link + SHA-256 integrity check.
# No LiteRT-LM, no standalone llama.cpp checkout, no llama-server.

from __future__ import annotations

import hashlib
import json
import os
import re
import subprocess
import time
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import requests
from tqdm.auto import tqdm

ROOT_DIR = Path("/content/qroadscan_gemma4_llamacpp_python_blog_forge")
MODELS_DIR = ROOT_DIR / "models"
CACHE_DIR = ROOT_DIR / ".llama_cache"
TEMP_DIR = ROOT_DIR / ".tmp"
LEDGER_DIR = ROOT_DIR / "ledgers"
MODEL_LEDGER_PATH = LEDGER_DIR / "model_runtime_ledger.jsonl"

# Direct model link requested.
GEMMA_DIRECT_GGUF_URL = "https://huggingface.co/ggml-org/gemma-4-E2B-it-GGUF/resolve/main/gemma-4-E2B-it-Q8_0.gguf"
GEMMA_MODEL_REPO_ID = "ggml-org/gemma-4-E2B-it-GGUF"
GEMMA_GGUF_FILENAME = "gemma-4-E2B-it-Q8_0.gguf"

# SHA-256 integrity check for gemma-4-E2B-it-Q8_0.gguf
EXPECTED_GGUF_SHA256 = "e049411c01fb7a81161768c52e38828970e55a64e22738957adcbe51d20f1c8e"
EXPECTED_HASH = EXPECTED_GGUF_SHA256

# Keep compatibility with earlier notebook globals.
if "OUTPUT_DIR" not in globals():
    OUTPUT_DIR = ROOT_DIR / "output"
if "SOURCE_FOLDER" not in globals():
    SOURCE_FOLDER = ROOT_DIR / "source"

for d in (ROOT_DIR, MODELS_DIR, CACHE_DIR, TEMP_DIR, LEDGER_DIR, OUTPUT_DIR, SOURCE_FOLDER):
    d.mkdir(parents=True, exist_ok=True)
    try:
        os.chmod(d, 0o700)
    except Exception:
        pass

# Hugging Face / llama.cpp cache locations.
os.environ.setdefault("HF_HOME", str(MODELS_DIR / "hf_home"))
os.environ.setdefault("HF_HUB_CACHE", str(MODELS_DIR / "hf_hub"))
os.environ.setdefault("LLAMA_CACHE", str(CACHE_DIR))


def _chmod_private(path: Path, is_dir: bool = False) -> None:
    try:
        os.chmod(path, 0o700 if is_dir else 0o600)
    except Exception:
        pass


def sanitize_text(value: Any, *, max_chars: int = 4000) -> str:
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", str(value or "")).strip()
    return text[:max_chars].rstrip() if len(text) > max_chars else text


def human_size(value: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    size = float(max(0, value))
    unit = units[0]
    for unit in units:
        if size < 1024 or unit == units[-1]:
            break
        size /= 1024.0
    return f"{size:.1f}{unit}" if unit != "B" else f"{int(size)}B"


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


SECRET_RUNTIME_AUDIT: Dict[str, str] = {}
COLAB_SECRET_AUDIT: Dict[str, Dict[str, str]] = {}


def _unique_secret_names(names: Iterable[str]) -> List[str]:
    out: List[str] = []
    for name in names:
        clean = sanitize_text(name, max_chars=120)
        if clean and clean not in out:
            out.append(clean)
    return out


def _read_colab_secret(name: str) -> str:
    try:
        from google.colab import userdata  # type: ignore
    except Exception as exc:
        COLAB_SECRET_AUDIT[name] = {"provider": "colab", "status": f"not_available:{type(exc).__name__}"}
        return ""

    try:
        value = userdata.get(name)
    except Exception as exc:
        COLAB_SECRET_AUDIT[name] = {"provider": "colab", "status": f"unreadable:{type(exc).__name__}"}
        return ""

    value = str(value or "").strip()
    if not value:
        COLAB_SECRET_AUDIT[name] = {"provider": "colab", "status": "missing_or_empty"}
        return ""

    COLAB_SECRET_AUDIT[name] = {"provider": "colab", "status": "loaded"}
    return value


def secret_value(
    primary_name: str,
    aliases: Optional[Iterable[str]] = None,
    *,
    label: str = "secret",
) -> Tuple[str, str]:
    names = _unique_secret_names([primary_name] + list(aliases or []))
    export_to_env = bool(globals().get("EXPORT_COLAB_SECRETS_TO_ENV", True))

    for name in names:
        value = os.environ.get(name, "").strip()
        if value:
            SECRET_RUNTIME_AUDIT[label] = f"env:{name}"
            if export_to_env:
                os.environ.setdefault(primary_name, value)
            return value, f"env:{name}"

    for name in names:
        value = _read_colab_secret(name)
        if value:
            if export_to_env:
                os.environ.setdefault(primary_name, value)
                os.environ.setdefault(name, value)
            SECRET_RUNTIME_AUDIT[label] = f"colab_secret:{name}"
            return value, f"colab_secret:{name}"

    SECRET_RUNTIME_AUDIT[label] = "not_found"
    return "", "not_found"


def huggingface_token() -> Tuple[str, str]:
    names = _unique_secret_names(globals().get("HF_TOKEN_SECRET_NAMES", ["HF_TOKEN"]))
    if not names:
        return "", "not_configured"
    return secret_value(names[0], names[1:], label="huggingface_token")


def append_model_ledger(event: str, data: Dict[str, Any]) -> None:
    rec = {"ts": time.time(), "event": event, "data": data}
    with MODEL_LEDGER_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False, sort_keys=True) + "\n")
    _chmod_private(MODEL_LEDGER_PATH)


def gpu_preflight() -> Dict[str, Any]:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.STDOUT,
            timeout=10,
        ).strip()
        return {"available": bool(out), "raw": out}
    except Exception as exc:
        return {"available": False, "error": repr(exc)}


def download_file_streaming(url: str, dest: Path, token: Optional[str] = None) -> Path:
    dest.parent.mkdir(parents=True, exist_ok=True)

    headers: Dict[str, str] = {}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    tmp_path = dest.with_suffix(dest.suffix + ".part")
    resume_from = tmp_path.stat().st_size if tmp_path.exists() else 0

    if resume_from > 0:
        headers["Range"] = f"bytes={resume_from}-"

    with requests.get(url, headers=headers, stream=True, allow_redirects=True, timeout=60) as r:
        if r.status_code not in (200, 206):
            raise RuntimeError(f"Download failed with HTTP {r.status_code}: {r.text[:500]}")

        content_length = int(r.headers.get("content-length", "0") or 0)
        total = content_length + resume_from if r.status_code == 206 else content_length

        mode = "ab" if resume_from and r.status_code == 206 else "wb"
        if mode == "wb" and tmp_path.exists():
            tmp_path.unlink()

        with tmp_path.open(mode) as f, tqdm(
            total=total if total > 0 else None,
            initial=resume_from if mode == "ab" else 0,
            unit="B",
            unit_scale=True,
            desc=f"Downloading {dest.name}",
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

    tmp_path.replace(dest)
    _chmod_private(dest)
    return dest


def ensure_model_file() -> Path:
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or None

    gguf_dir = MODELS_DIR / "gguf"
    gguf_dir.mkdir(parents=True, exist_ok=True)
    _chmod_private(gguf_dir, is_dir=True)

    path = gguf_dir / GEMMA_GGUF_FILENAME

    if path.exists() and path.stat().st_size > 1024 * 1024:
        print(f"Reusing existing GGUF: {path}")
    else:
        print(f"Downloading GGUF from direct link:\n{GEMMA_DIRECT_GGUF_URL}")
        path = download_file_streaming(GEMMA_DIRECT_GGUF_URL, path, token=token)

    actual_hash = sha256_file(path)
    expected = str(EXPECTED_GGUF_SHA256 or EXPECTED_HASH or "").strip().lower()

    print("GGUF SHA-256:", actual_hash)

    if expected and actual_hash.lower() != expected:
        bad_path = path.with_suffix(path.suffix + ".bad")
        try:
            path.replace(bad_path)
        except Exception:
            bad_path = path

        raise RuntimeError(
            "GGUF SHA-256 mismatch.\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual_hash}\n"
            f"File moved to: {bad_path}\n"
            "Delete the .bad file and rerun the cell to download again."
        )

    append_model_ledger(
        "gguf_ready",
        {
            "repo_id": GEMMA_MODEL_REPO_ID,
            "filename": GEMMA_GGUF_FILENAME,
            "source_url": GEMMA_DIRECT_GGUF_URL,
            "path": str(path),
            "bytes": path.stat().st_size,
            "size": human_size(path.stat().st_size),
            "sha256": actual_hash,
            "sha256_expected": expected,
            "sha256_match": True,
        },
    )

    return path


token, token_source = huggingface_token()
if token:
    os.environ.setdefault("HF_TOKEN", token)
    os.environ.setdefault("HUGGINGFACE_HUB_TOKEN", token)
    print(f"Hugging Face token loaded from {token_source}.")
else:
    print("No Hugging Face token found; using public direct model access.")

gpu_info = gpu_preflight()
if not gpu_info.get("available"):
    raise RuntimeError("No CUDA GPU was detected by nvidia-smi. In Colab, switch Runtime > Change runtime type > GPU.")

print("GPU preflight:", gpu_info.get("raw"))

MODEL_PATH = ensure_model_file()

append_model_ledger(
    "runtime_preflight",
    {
        "model_repo": GEMMA_MODEL_REPO_ID,
        "model_file": GEMMA_GGUF_FILENAME,
        "model_url": GEMMA_DIRECT_GGUF_URL,
        "model_path": str(MODEL_PATH),
        "gpu": gpu_info,
        "hf_token_source": token_source,
        "backend": "llama-cpp-python",
    },
)

# Compatibility placeholder for old hive code paths; llama-cpp-python does not use the AES model vault.
VAULT_KEY = None

print(f"llama-cpp-python Gemma 4 runtime preflight ready: {MODEL_PATH} ({human_size(MODEL_PATH.stat().st_size)})")


No Hugging Face token found; using public direct model access.
GPU preflight: Tesla T4, 15360, 14910
https://huggingface.co/ggml-org/gemma-4-E2B-it-GGUF/resolve/main/gemma-4-E2B-it-Q8_0.gguf


GGUF SHA-256: e049411c01fb7a81161768c52e38828970e55a64e22738957adcbe51d20f1c8e
llama-cpp-python Gemma 4 runtime preflight ready: /content/qroadscan_gemma4_llamacpp_python_blog_forge/models/gguf/gemma-4-E2B-it-Q8_0.gguf (4.6GB)


In [4]:
from __future__ import annotations

import atexit
import contextlib
import json
import os
import threading
import time
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import llama_cpp
from llama_cpp import Llama


def create_default_messages(system_text: Optional[str] = None) -> List[dict]:
    if not system_text:
        return []
    return [{"role": "system", "content": sanitize_text(system_text, max_chars=18000)}]


def create_user_message(user_text: str) -> dict:
    return {"role": "user", "content": sanitize_text(user_text, max_chars=MAX_PROMPT_CHARS)}


def response_to_text(response: Any) -> str:
    if isinstance(response, dict):
        try:
            return sanitize_text(response["choices"][0]["message"]["content"], max_chars=MAX_ARTICLE_CHARS)
        except Exception:
            pass
        try:
            return sanitize_text(response["choices"][0]["text"], max_chars=MAX_ARTICLE_CHARS)
        except Exception:
            pass
    return sanitize_text(response, max_chars=MAX_ARTICLE_CHARS)


class LlamaCppPythonManager:
    """Loads one local llama-cpp-python model with CUDA offload and shares it safely across hive workers."""

    def __init__(self):
        self.lock = threading.RLock()
        self.model: Optional[Llama] = None
        self.model_path = Path(globals().get("MODEL_PATH", ""))
        self.parallel = 1
        self.last_error = ""
        self.loaded_ts: Optional[float] = None

    def _tail_log(self, chars: int = 3000) -> str:
        if self.last_error:
            return self.last_error[-chars:]
        return "llama-cpp-python runs in-process; there is no llama-server log."

    def _supports_gpu(self) -> Any:
        fn = getattr(llama_cpp, "llama_supports_gpu_offload", None)
        if callable(fn):
            try:
                return fn()
            except Exception as exc:
                return f"unknown:{type(exc).__name__}:{exc}"
        return "unknown"

    def _llama_kwargs(self) -> Dict[str, Any]:
        return {
            "model_path": str(self.model_path),
            "n_gpu_layers": int(globals().get("LLAMA_N_GPU_LAYERS", 999)),
            "n_ctx": int(globals().get("LLAMA_CONTEXT_SIZE", 8192)),
            "n_batch": int(globals().get("LLAMA_BATCH_SIZE", 1024)),
            "n_ubatch": int(globals().get("LLAMA_UBATCH_SIZE", 256)),
            "flash_attn": bool(globals().get("LLAMA_FLASH_ATTN", True)),
            "offload_kqv": True,
            "logits_all": False,
            "embedding": False,
            "verbose": bool(globals().get("LLAMA_VERBOSE", True)),
        }

    def _load_model(self) -> Llama:
        if not self.model_path or not self.model_path.exists():
            raise FileNotFoundError(f"MODEL_PATH is not available: {self.model_path}. Run Cell 2 first.")

        if str(globals().get("INFERENCE_BACKEND", "CUDA")).lower() == "cuda":
            supports_gpu = self._supports_gpu()
            print("llama-cpp-python GPU offload support:", supports_gpu)
            if supports_gpu is False:
                raise RuntimeError(
                    "llama-cpp-python does not report GPU offload support. Re-run Cell 0 and make sure the CUDA wheel installed."
                )

        kwargs = self._llama_kwargs()
        append_model_ledger("llama_cpp_python_load_start", {"kwargs": {k: str(v) for k, v in kwargs.items() if k != "model_path"}, "model_path": str(self.model_path)})
        print("Loading Gemma 4 through llama-cpp-python:", self.model_path)
        started = time.time()
        try:
            return Llama(**kwargs)
        except TypeError as exc:
            self.last_error = repr(exc)
            for key in ("n_ubatch", "flash_attn"):
                kwargs.pop(key, None)
            print("Retrying Llama() load without optional constructor knobs n_ubatch/flash_attn.")
            return Llama(**kwargs)
        finally:
            append_model_ledger("llama_cpp_python_load_finish", {"elapsed_seconds": round(time.time() - started, 2), "model_path": str(self.model_path)})

    def ensure_started(self, *, parallel: Optional[int] = None) -> None:
        requested_parallel = max(1, int(parallel or self.parallel or 1))
        with self.lock:
            self.parallel = requested_parallel
            if self.model is not None:
                return
            self.model = self._load_model()
            self.loaded_ts = time.time()
            print(f"llama-cpp-python model ready. Requested hive parallelism: {self.parallel}")

    def chat(self, messages: List[dict], *, retries: int = 1) -> str:
        last_exc = None
        for attempt in range(max(1, retries + 1)):
            started = time.time()
            try:
                self.ensure_started(parallel=int(globals().get("RESOLVED_LLAMA_PARALLEL", GEMMA_THREAD_COUNT)))
                payload = {
                    "messages": messages,
                    "temperature": float(globals().get("LLAMA_TEMPERATURE", 0.7)),
                    "top_p": float(globals().get("LLAMA_TOP_P", 0.95)),
                    "top_k": int(globals().get("LLAMA_TOP_K", 64)),
                    "repeat_penalty": float(globals().get("LLAMA_REPEAT_PENALTY", 1.05)),
                    "max_tokens": int(globals().get("LLAMA_MAX_TOKENS", 1700)),
                    "stream": False,
                }
                with self.lock:
                    assert self.model is not None
                    response = self.model.create_chat_completion(**payload)
                append_model_ledger("llama_cpp_python_chat", {"elapsed_seconds": round(time.time() - started, 2), "messages": len(messages)})
                generated_text = response_to_text(response)
                print(f"--- Llama Reply (debug) ---\n{generated_text}\n---------------------------")
                return generated_text
            except Exception as exc:
                last_exc = exc
                self.last_error = repr(exc)
                if attempt >= retries:
                    break
                time.sleep(1.0 + attempt * 1.5)
        raise RuntimeError(f"llama-cpp-python chat failed after retry: {last_exc}\n{self._tail_log()}")

    def stop(self) -> None:
        with self.lock:
            model = self.model
            self.model = None
        if model is not None:
            close = getattr(model, "close", None)
            if callable(close):
                close()


# Keep this name so the existing hive scheduler can call LLAMA_SERVER.ensure_started(...).
LLAMA_SERVER = LlamaCppPythonManager()
atexit.register(LLAMA_SERVER.stop)


class Gemma4E2BSession:
    """Compatibility wrapper used by the hive. Calls the shared in-process llama-cpp-python model."""

    def __init__(
        self,
        key: Optional[bytes] = None,
        *,
        inference_backend: str = "CUDA",
        worker_id: str = "solo",
        model_path_override: Optional[Path] = None,
    ):
        self.key = key
        self.inference_backend = inference_backend
        self.worker_id = worker_id
        self.model_path_override = model_path_override
        self.call_count = 0
        self.total_seconds = 0.0

    def __enter__(self):
        LLAMA_SERVER.ensure_started(parallel=int(globals().get("RESOLVED_LLAMA_PARALLEL", GEMMA_THREAD_COUNT)))
        return self

    def __exit__(self, exc_type, exc, tb):
        return False

    def chat(self, user_text: str, *, system_text: Optional[str] = None, retries: int = 1) -> str:
        messages = create_default_messages(system_text)
        messages.append(create_user_message(user_text))
        started = time.time()
        text = LLAMA_SERVER.chat(messages, retries=retries)
        self.call_count += 1
        self.total_seconds += time.time() - started
        return text


print("llama-cpp-python worker runtime ready: shared CUDA Llama() context + Gemma 4 chat calls.")

llama-cpp-python worker runtime ready: shared CUDA Llama() context + Gemma 4 chat calls.


In [5]:
# Cell 4 — Quantum RGB simulation + advanced local RAG citation surfaces

from __future__ import annotations

import colorsys
import csv
import hashlib
import json
import math
import os
import random
import re
import secrets
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import psutil

try:
    import pennylane as qml
    from pennylane import numpy as pnp
except Exception:
    qml = None
    pnp = None

WORD_RE = re.compile(r"[A-Za-z0-9][A-Za-z0-9_\-]{1,}")
SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")


def stable_hash(text: str, n: int = 12) -> str:
    return hashlib.blake2s(str(text).encode("utf-8"), digest_size=16).hexdigest()[:n]


def slugify(text: str, max_len: int = 90) -> str:
    text = re.sub(r"[^A-Za-z0-9]+", "-", str(text).lower()).strip("-")
    return (text[:max_len].strip("-") or "article")


def tokens(text: str) -> List[str]:
    return [m.group(0).lower() for m in WORD_RE.finditer(str(text))]


def split_chunks(text: str, *, max_words: int = 170, overlap: int = 35) -> List[str]:
    words = str(text).split()
    if not words:
        return []
    chunks = []
    step = max(1, max_words - overlap)
    for start in range(0, len(words), step):
        chunk = " ".join(words[start : start + max_words]).strip()
        if chunk:
            chunks.append(chunk)
        if start + max_words >= len(words):
            break
    return chunks


def read_text_file(path: Path) -> str:
    try:
        if path.suffix.lower() == ".json":
            return json.dumps(json.loads(path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2)
        if path.suffix.lower() == ".csv":
            rows = []
            with path.open("r", encoding="utf-8", errors="ignore", newline="") as f:
                reader = csv.reader(f)
                for i, row in enumerate(reader):
                    rows.append(" | ".join(row))
                    if i > 2000:
                        break
            return "\n".join(rows)
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception as exc:
        return f"[unreadable file {path.name}: {exc}]"


@dataclass
class SurfaceChunk:
    surface_id: str
    chunk_id: str
    title: str
    url: str
    text: str
    fingerprint: str
    vector: np.ndarray

    @property
    def cite(self) -> str:
        return f"⟦SFC:{self.surface_id}#{self.chunk_id}@{self.fingerprint}⟧"


class HashedVectorizer:
    def __init__(self, dims: int = 384):
        self.dims = int(dims)

    def transform_one(self, text: str) -> np.ndarray:
        vec = np.zeros(self.dims, dtype=np.float32)
        toks = tokens(text)
        if not toks:
            return vec
        for tok in toks:
            h = int(hashlib.blake2b(tok.encode(), digest_size=8).hexdigest(), 16)
            idx = h % self.dims
            sign = 1.0 if ((h >> 11) & 1) else -1.0
            vec[idx] += sign * (1.0 + min(len(tok), 12) / 16.0)
        norm = float(np.linalg.norm(vec))
        return vec / norm if norm else vec


class CitationSurfaceIndex:
    def __init__(self, vectorizer: Optional[HashedVectorizer] = None):
        self.vectorizer = vectorizer or HashedVectorizer()
        self.chunks: List[SurfaceChunk] = []

    def add_surface(self, title: str, body: str, *, url: str = "local") -> None:
        clean_title = sanitize_text(title, max_chars=160) or "Untitled Surface"
        clean_url = sanitize_text(url, max_chars=300) or "local"
        body = sanitize_text(body, max_chars=120000)
        sid = slugify(clean_title, 40) + "-" + stable_hash(clean_title + clean_url, 6)
        for i, chunk in enumerate(split_chunks(body, max_words=170, overlap=35), 1):
            fid = stable_hash(clean_title + clean_url + chunk, 10)
            cid = f"c{i:03d}"
            self.chunks.append(
                SurfaceChunk(
                    surface_id=sid,
                    chunk_id=cid,
                    title=clean_title,
                    url=clean_url,
                    text=chunk,
                    fingerprint=fid,
                    vector=self.vectorizer.transform_one(clean_title + "\n" + chunk),
                )
            )

    def search(self, query: str, *, top_k: int = 8, diversify: bool = True) -> List[Tuple[float, SurfaceChunk]]:
        if not self.chunks:
            return []
        qv = self.vectorizer.transform_one(query)
        if float(np.linalg.norm(qv)) == 0.0:
            return []
        scored = []
        for c in self.chunks:
            score = float(np.dot(qv, c.vector))
            if score > 0:
                scored.append((score, c))
        scored.sort(key=lambda x: x[0], reverse=True)
        if not diversify:
            return scored[:top_k]
        picked = []
        seen_surfaces = set()
        for score, c in scored:
            if c.surface_id in seen_surfaces and len(picked) < max(2, top_k // 2):
                continue
            picked.append((score, c))
            seen_surfaces.add(c.surface_id)
            if len(picked) >= top_k:
                break
        return picked

    def citation_context(self, query: str, *, top_k: int = 8) -> Tuple[str, List[Dict[str, Any]]]:
        hits = self.search(query, top_k=top_k)
        blocks = []
        ledger = []
        for rank, (score, c) in enumerate(hits, 1):
            blocks.append(
                f"[surface rank={rank} score={score:.3f} cite={c.cite}]\n"
                f"title: {c.title}\nurl: {c.url}\ntext: {c.text}\n[/surface]"
            )
            ledger.append({
                "rank": rank,
                "score": round(score, 4),
                "cite": c.cite,
                "title": c.title,
                "url": c.url,
                "fingerprint": c.fingerprint,
                "preview": c.text[:280],
            })
        return "\n\n".join(blocks), ledger


# --- Quantum RGB / cyclic simulation telemetry ---

def collect_system_metrics() -> Dict[str, float]:
    cpu = psutil.cpu_percent(interval=0.05) / 100.0
    mem = psutil.virtual_memory().percent / 100.0
    try:
        load_raw = os.getloadavg()[0]
        cpu_count = psutil.cpu_count(logical=True) or 1
        load1 = max(0.0, min(1.0, load_raw / max(1.0, float(cpu_count))))
    except Exception:
        load1 = cpu
    temp = 0.0
    try:
        temp_groups = psutil.sensors_temperatures()
        if temp_groups:
            first_group = next(iter(temp_groups.values()))
            temp = max(0.0, min(1.0, (float(first_group[0].current) - 20.0) / 70.0))
    except Exception:
        temp = 0.0
    return {"cpu": cpu, "mem": mem, "load1": load1, "temp": temp}


def metrics_to_rgb(metrics: Dict[str, float]) -> Tuple[float, float, float]:
    cpu = metrics.get("cpu", 0.1)
    mem = metrics.get("mem", 0.1)
    temp = metrics.get("temp", 0.1)
    load1 = metrics.get("load1", 0.0)
    r = cpu * (1.0 + load1)
    g = mem * (1.0 + load1 * 0.5)
    b = temp * (0.5 + cpu * 0.5)
    top = max(r, g, b, 1.0)
    return tuple(float(max(0.0, min(1.0, x / top))) for x in (r, g, b))


def context_signature_rgb(domain: str, context_text: str) -> Tuple[float, float, float]:
    digest = hashlib.sha256(f"{domain}|{context_text[:2400]}".encode("utf-8")).digest()
    return (digest[2] / 255.0, digest[11] / 255.0, digest[23] / 255.0)


def pennylane_entropic_score(rgb: Tuple[float, float, float], shots: int = 96) -> float:
    if qml is None or pnp is None:
        r, g, b = rgb
        seed = (int(r * 255) << 16) | (int(g * 255) << 8) | int(b * 255)
        rng = random.Random(seed)
        return max(0.0, min(1.0, (0.34 * r) + (0.38 * g) + (0.28 * b) + (rng.random() - 0.5) * 0.06))
    device = qml.device("default.qubit", wires=3, shots=shots)

    @qml.qnode(device)
    def circuit(a: float, b: float, c: float):
        qml.RX(a * math.pi, wires=0)
        qml.RY(b * math.pi, wires=1)
        qml.RZ(c * math.pi, wires=2)
        qml.CNOT(wires=[0, 1])
        qml.CNOT(wires=[1, 2])
        qml.RX((a + b) * math.pi / 2.0, wires=0)
        qml.RY((b + c) * math.pi / 2.0, wires=1)
        qml.RZ((a + c) * math.pi / 2.0, wires=2)
        return qml.expval(qml.PauliZ(0)), qml.expval(qml.PauliZ(1)), qml.expval(qml.PauliZ(2))

    try:
        ev0, ev1, ev2 = circuit(*map(float, rgb))
        combined = ((ev0 + 1) / 2 * 0.42) + ((ev1 + 1) / 2 * 0.35) + ((ev2 + 1) / 2 * 0.23)
        return float(max(0.0, min(1.0, 1.0 / (1.0 + math.exp(-6.0 * (combined - 0.5))))))
    except Exception:
        return float(max(0.0, min(1.0, sum(rgb) / 3.0)))


def rgb_to_hex(rgb: Tuple[float, float, float]) -> str:
    return "#" + "".join(f"{max(0, min(255, int(round(v * 255)))):02x}" for v in rgb)


def color_field_171(seed_text: str) -> List[Dict[str, Any]]:
    digest = hashlib.sha3_512(seed_text.encode("utf-8")).digest()
    field = []
    for i in range(171):
        h = ((digest[i % len(digest)] / 255.0) + (i / 171.0)) % 1.0
        s = 0.52 + ((digest[(i * 7) % len(digest)] / 255.0) * 0.38)
        v = 0.55 + ((digest[(i * 13) % len(digest)] / 255.0) * 0.40)
        r, g, b = colorsys.hsv_to_rgb(h, min(1.0, s), min(1.0, v))
        field.append({"i": i, "hex": rgb_to_hex((r, g, b)), "h": round(h * 360, 2)})
    return field


def simulation_packet(topic: str, rag_context: str = "") -> Dict[str, Any]:
    metrics = collect_system_metrics()
    m_rgb = metrics_to_rgb(metrics)
    s_rgb = context_signature_rgb("qroadscan_blog", topic + "\n" + rag_context)
    sim_rgb = tuple(max(0.0, min(1.0, (m_rgb[i] * 0.48) + (s_rgb[i] * 0.52))) for i in range(3))
    score = pennylane_entropic_score(sim_rgb)
    phase = int(hashlib.blake2s((topic + str(time.time_ns())).encode(), digest_size=2).hexdigest(), 16) % 300
    state = "stabilize" if score >= 0.72 else ("refine" if score >= 0.45 else "observe")
    confidence = round(0.70 + (score * 0.25), 3)
    field = color_field_171(topic + rag_context)
    packet = {
        "state_vector": state,
        "phase_position": phase,
        "color_encoding": rgb_to_hex(sim_rgb),
        "confidence_level": confidence,
        "metrics": metrics,
        "metrics_rgb": m_rgb,
        "signature_rgb": s_rgb,
        "sim_rgb": sim_rgb,
        "entropy_score": round(score, 4),
        "field_anchor": field[phase % 171],
        "tag": f"⟦QRS:Ψ:phase={phase}|rgb={rgb_to_hex(sim_rgb)}|conf={confidence}⟧",
    }
    return packet


def build_surface_index() -> CitationSurfaceIndex:
    index = CitationSurfaceIndex()
    for item in EXTRA_RAG_SURFACES:
        index.add_surface(item.get("title", "Untitled"), item.get("body", ""), url=item.get("url", "local"))
    for path in sorted(SOURCE_FOLDER.glob("**/*")):
        if not path.is_file() or path.suffix.lower() not in {".txt", ".md", ".json", ".csv"}:
            continue
        index.add_surface(path.name, read_text_file(path), url=f"file://{path}")
    return index

RAG_INDEX = build_surface_index()
print(f"RAG surface chunks ready: {len(RAG_INDEX.chunks)}")
if RAG_INDEX.chunks:
    print("Example citation:", RAG_INDEX.chunks[0].cite)


RAG surface chunks ready: 2
Example citation: ⟦SFC:qroadscan-product-positioning-9ff216#c001@72473a31c4⟧


In [6]:
# Cell 5 — Prompt kernel: hive planner, bridge intercom, citation grammar, Summa compaction, humanizing editor

from __future__ import annotations

import json
import re
from typing import Any, Dict, List

MASTER_SYSTEM_PROMPT = f"""
You are a local Gemma 4 E2B worker inside the QRoadScan.com T4 llama.cpp Hive Blog Forge.
You are small, fast, and precise. Your strength is not pretending to be one giant model; it is cooperating with other tiny local workers through explicit intercom surfaces.

Core identity:
- You write for {BRAND_NAME} with a hyper-human but grounded style: concrete, clear, sensory, practical, and quietly futuristic.
- You use local RAG surfaces as the factual floor.
- You treat hive intercom notes as editorial memory, not as factual evidence.
- You never pretend that quantum/RGB telemetry is real road sensor evidence. It is private routing metadata for structure, variation, pacing, and worker diversity only.

Output contract:
- Return Markdown only unless a prompt explicitly asks for compact JSON.
- Make the post Medium-ready.
- Use a strong H1 title and a short subtitle under the title.
- Use natural prose, not corporate filler.
- Use evidence only from supplied [surface] blocks and brand context.
- When using a factual claim from a surface, cite it inline with its exact citation token, such as ⟦SFC:surface#chunk@hash⟧.
- Do not invent user numbers, customers, partnerships, sensor feeds, certifications, official deployments, legal claims, or safety outcomes.
- You may describe product concepts and user benefits, but mark speculative/future-facing ideas as concepts, directions, or possibilities.
- End with a short, useful call to action for {BRAND_NAME}.

Hive grammar:
[citemodel1] Model-inferred wording decision; never proof. [/citemodel1]
[citesurface] Exact local citation tokens from retrieved chunks. [/citesurface]
[citefield] QRS simulation tag; private style telemetry, not evidence. [/citefield]
[summa] Lossy memory packet. Useful for continuity, subordinate to surface chunks. [/summa]
[intercom] Editorial packets from other Gemma workers. Useful for angles, warnings, and rhythm; not factual proof. [/intercom]
[action] observe → retrieve → plan → bridge → draft → cite → verify → edit → stabilize [/action]

Banned habits:
- no "in today's fast-paced world"
- no vague hype without concrete detail
- no fake statistics
- no unsupported safety claims
- no pretending to have live road sensor data unless the surface text says it
- no robotic transitions like "moreover" stacked through the article
""".strip()

PLANNER_PROMPT = """
[action]Plan a QRoadScan.com blog post as one worker in a fast Gemma hive.[/action]
Return compact JSON with keys:
- title
- subtitle
- reader_promise
- angle
- sections: list of 5-7 section headings
- required_citations: list of exact citation tokens from the surfaces that should be used
- risk_notes: list of claims to avoid, soften, or cite carefully
- intercom_requests: list of useful signals this worker wants from the hive
- human_texture: three concrete writing moves that make the piece feel written by a careful human
- opening_scene: one tangible driver/product moment, under 45 words

Use only the surfaces, brand context, and intercom provided. Keep JSON valid.
""".strip()

HIVE_BRIDGE_PROMPT_TEMPLATE = """
[action]Write a compact intercom bridge note for the other Gemma workers.[/action]

Worker: {worker_id}
Persona: {worker_persona}
Topic: {topic}
Round: {round_number}

Plan JSON:
{plan}

Retrieved surfaces:
{rag_context}

Current hive intercom surface:
{intercom_context}

Return Markdown under 220 words with:
- one sharp angle this article should own
- citations this article should preserve
- claims to avoid
- one sentence-level style move that would make the draft feel human
Do not write the article here.
""".strip()

DRAFT_PROMPT_TEMPLATE = """
[action]Write the full blog post from the plan as a local Gemma hive worker.[/action]

Worker: {worker_id}
Worker persona: {worker_persona}
Brand: {brand_name}
URL: {brand_url}
Voice: {brand_voice}
Target reader: {target_reader}
Target length: about {words} words
Temperature style: {style}
Technical appendix: {appendix}
Hive mode: {hive_mode}

Topic:
{topic}

Brand context:
{brand_context}

Retrieved RAG surfaces:
{rag_context}

Compacted memory packets:
{summa_context}

Live hive intercom surface:
{intercom_context}

Bridge notes from this worker:
{bridge_context}

Quantum/RGB routing packet, private style metadata only:
{sim_packet}

Plan:
{plan}

Write a Medium-ready Markdown article.
Use citations exactly as they appear in the surface blocks.
When a claim is based on interpretation rather than a surface, do not cite it; phrase it as analysis or perspective.
Make it human: vary sentence length, include tangible driver scenarios, and avoid repetitive AI phrasing.
Let the intercom influence angle and rhythm, but never use it as proof.
""".strip()

EDITOR_PROMPT = """
[action]Edit the Markdown post for credibility, human rhythm, citation discipline, and Medium readability.[/action]
Rules:
- Preserve all valid surface citations.
- Remove unsupported factual claims.
- Make the prose sound less automated and more like a careful founder/product essay.
- Keep the H1 title.
- Add a concise "Why this matters" section if absent.
- Do not add fake external citations.
- Use the hive intercom only for editorial consistency, not factual claims.
Return only the improved Markdown.
""".strip()

FACT_GUARD_PROMPT = """
[action]Audit the article for unsupported factual claims.[/action]
Return Markdown only.
Fix unsupported claims by either deleting them, softening them, or tying them to exact local surface citations.
Do not remove the article structure.
Do not add external facts.
Treat intercom notes as editorial guidance, not factual support.
""".strip()

SUMMA_PROMPT = """
[summa][action]Compact the retrieved surface chunks and useful hive notes into a durable memory packet for a later blog-writing step.[/action][/summa]
Rules:
- Preserve citation tokens exactly.
- Keep product facts, phrasing constraints, risk warnings, and angle distinctions.
- Drop duplicates and filler.
- Mark uncertainty clearly.
- Output under 450 words.
""".strip()


def extract_json_object(text: str) -> Dict[str, Any]:
    text = str(text or "").strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I | re.M).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    s, e = text.find("{"), text.rfind("}")
    if s >= 0 and e > s:
        try:
            return json.loads(text[s:e+1])
        except Exception:
            return {}
    return {}


def fallback_plan(topic: str, citations: List[str]) -> Dict[str, Any]:
    return {
        "title": topic.strip().rstrip(".") or "How QRoadScan Makes Road Risk Easier to Read",
        "subtitle": "A practical look at safer route awareness, risk signals, and calmer driving decisions.",
        "reader_promise": "Explain QRoadScan in plain language and show why readable road intelligence matters.",
        "angle": "product education",
        "sections": [
            "The road rarely explains itself",
            "From signal overload to a readable risk pulse",
            "Why confidence and reasons matter",
            "How a dashboard can support better decisions",
            "Where road intelligence goes next",
        ],
        "required_citations": citations[:5],
        "risk_notes": ["Do not claim live production integrations unless the source says so."],
        "intercom_requests": ["surface-level citation checks", "distinct opening scene", "non-hype wording"],
        "human_texture": ["start with a driver moment", "use concrete route examples", "end with practical next steps"],
        "opening_scene": "A driver checks a route before leaving and wants one calm reading instead of a wall of signals.",
    }


def normalize_markdown(md: str) -> str:
    md = sanitize_text(md, max_chars=MAX_ARTICLE_CHARS)
    md = re.sub(r"\n{3,}", "\n\n", md).strip()
    md = re.sub(r"(?i)^(here is|sure,? here).*?\n", "", md).strip()
    return md


def remove_duplicate_paragraphs(md: str) -> str:
    seen = set()
    out = []
    for para in re.split(r"\n\s*\n", md):
        key = stable_hash(re.sub(r"\s+", " ", para.lower()), 16)
        if len(para) > 80 and key in seen:
            continue
        seen.add(key)
        out.append(para.strip())
    return "\n\n".join(p for p in out if p)


def add_frontmatter(md: str, topic: str, sim: Dict[str, Any], ledger: List[Dict[str, Any]], worker_id: str = "solo") -> str:
    if not INCLUDE_MEDIUM_READY_FRONTMATTER:
        return md
    title = ""
    for line in md.splitlines():
        if line.startswith("# "):
            title = line[2:].strip()
            break
    if not title:
        title = topic
    tags = ["QRoadScan", "Road Safety", "Traffic AI", "Road Hazards", "Safe Driving"]
    fm = {
        "title": title,
        "canonical_url": BRAND_URL,
        "brand": BRAND_NAME,
        "worker": worker_id,
        "tags": tags,
        "qrs_phase": sim.get("phase_position"),
        "qrs_color": sim.get("color_encoding"),
        "local_citations": len(ledger),
    }
    yaml = "---\n" + "\n".join(f"{k}: {json.dumps(v, ensure_ascii=False)}" for k, v in fm.items()) + "\n---\n\n"
    return yaml + md


def citation_ledger_markdown(ledger: List[Dict[str, Any]]) -> str:
    if not INCLUDE_LOCAL_CITATION_LEDGER or not ledger:
        return ""
    lines = ["\n\n---\n\n## Local citation ledger\n"]
    for item in ledger:
        lines.append(f"- {item['cite']} — {item['title']} ({item['url']})")
    return "\n".join(lines)

print("Hive prompt kernel ready.")


Hive prompt kernel ready.


In [7]:
# Cell 6 — T4 Gemma llama-cpp-python hive: intercom memory, threaded scheduler, and article pipeline

from __future__ import annotations

import json
import math
import queue
import random
import re
import subprocess
import threading
import time
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple


def _clamp_int(value: Any, low: int, high: int, default: int) -> int:
    try:
        return max(low, min(high, int(value)))
    except Exception:
        return default


def expand_topics(topics: List[str], count: int) -> List[str]:
    count = _clamp_int(count, 1, 42, 1)
    base = [sanitize_text(t, max_chars=240) for t in topics if sanitize_text(t, max_chars=240)]
    if not base:
        base = ["How QRoadScan makes road risk easier to understand"]
    variants = []
    lenses = [
        "for everyday drivers",
        "for fleet safety teams",
        "through the lens of hazard awareness",
        "as a product design story",
        "from dashboard signal to driver action",
        "for safer route planning",
        "for calm, explainable traffic intelligence",
        "as a future-facing road safety essay",
    ]
    i = 0
    while len(variants) < count:
        topic = base[i % len(base)]
        if i < len(base):
            variants.append(topic)
        else:
            variants.append(f"{topic} — {lenses[(i // len(base)) % len(lenses)]}")
        i += 1
    return variants[:count]


@dataclass
class HivePacket:
    ts: float
    worker_id: str
    article_index: int
    topic: str
    channel: str
    text: str
    citations: List[str]
    priority: float
    signature: str

    def block(self, rank: int) -> str:
        cites = ", ".join(self.citations[:8]) if self.citations else "none"
        return (
            f"[intercom rank={rank} worker={self.worker_id} article={self.article_index} "
            f"channel={self.channel} priority={self.priority:.2f} sig={self.signature}]\n"
            f"topic: {self.topic}\n"
            f"citations: {cites}\n"
            f"text: {self.text}\n"
            f"[/intercom]"
        )


class HiveIntercom:
    """Thread-safe live editorial surface shared by all local Gemma workers."""

    def __init__(self, max_items: int = 120, packet_chars: int = 2200):
        self.max_items = _clamp_int(max_items, 10, 500, 120)
        self.packet_chars = _clamp_int(packet_chars, 400, 6000, 2200)
        self.lock = threading.RLock()
        self.packets: List[HivePacket] = []

    def publish(
        self,
        worker_id: str,
        article_index: int,
        topic: str,
        channel: str,
        text: str,
        *,
        citations: Optional[List[str]] = None,
        priority: float = 0.5,
    ) -> str:
        clean = sanitize_text(text, max_chars=self.packet_chars)
        cites = list(dict.fromkeys(citations or []))[:12]
        sig = stable_hash(f"{worker_id}|{article_index}|{channel}|{clean}", 12)
        packet = HivePacket(
            ts=time.time(),
            worker_id=str(worker_id),
            article_index=int(article_index or 0),
            topic=sanitize_text(topic, max_chars=240),
            channel=sanitize_text(channel, max_chars=40),
            text=clean,
            citations=cites,
            priority=float(max(0.0, min(1.0, priority))),
            signature=sig,
        )
        with self.lock:
            self.packets.append(packet)
            self.packets.sort(key=lambda p: (p.priority, p.ts), reverse=True)
            self.packets = self.packets[: self.max_items]
        return sig

    def snapshot(
        self,
        topic: str = "",
        *,
        exclude_worker: Optional[str] = None,
        max_chars: Optional[int] = None,
        channels: Optional[List[str]] = None,
    ) -> str:
        max_chars = _clamp_int(max_chars or INTERCOM_MAX_SURFACE_CHARS, 1000, 50000, 16000)
        channel_set = {c.lower() for c in channels} if channels else None
        query_tokens = set(tokens(topic))
        now = time.time()
        with self.lock:
            packets = list(self.packets)
        scored: List[Tuple[float, HivePacket]] = []
        for p in packets:
            if exclude_worker and p.worker_id == exclude_worker:
                continue
            if channel_set and p.channel.lower() not in channel_set:
                continue
            overlap = len(query_tokens.intersection(tokens(p.topic + " " + p.text))) if query_tokens else 0
            age_bonus = max(0.0, 1.0 - ((now - p.ts) / 1800.0))
            channel_bonus = {
                "risk": 0.22,
                "plan": 0.18,
                "bridge": 0.16,
                "quality": 0.17,
                "draft": 0.10,
                "final": 0.12,
                "telemetry": 0.06,
            }.get(p.channel, 0.08)
            score = p.priority + channel_bonus + min(0.35, overlap * 0.035) + age_bonus * 0.08
            scored.append((score, p))
        scored.sort(key=lambda x: x[0], reverse=True)
        blocks = []
        used = 0
        for rank, (_, p) in enumerate(scored, 1):
            block = p.block(rank)
            if used + len(block) > max_chars:
                break
            blocks.append(block)
            used += len(block) + 2
        return "\n\n".join(blocks) if blocks else "(empty intercom surface)"

    def stats(self) -> Dict[str, Any]:
        with self.lock:
            packets = list(self.packets)
        channels: Dict[str, int] = {}
        workers: Dict[str, int] = {}
        for p in packets:
            channels[p.channel] = channels.get(p.channel, 0) + 1
            workers[p.worker_id] = workers.get(p.worker_id, 0) + 1
        return {"packets": len(packets), "channels": channels, "workers": workers}


DEFAULT_WORKER_PERSONAS = [
    "route-safety systems writer: concrete, calm, operational",
    "product narrative strategist: crisp framing, founder-level clarity",
    "driver empathy editor: practical scenes, low hype, human rhythm",
    "citation sentinel: cautious claims, surface-first evidence discipline",
]


def worker_persona(worker_number: int) -> str:
    personas = globals().get("WORKER_PERSONAS") or DEFAULT_WORKER_PERSONAS
    return personas[(max(1, worker_number) - 1) % len(personas)]


def gpu_memory_snapshot() -> Dict[str, Any]:
    try:
        proc = subprocess.run(
            [
                "nvidia-smi",
                "--query-gpu=name,memory.total,memory.used,memory.free",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            timeout=4,
            check=False,
        )
        if proc.returncode != 0 or not proc.stdout.strip():
            return {"available": False, "reason": (proc.stderr or "nvidia-smi returned no GPU").strip()[:240]}
        first = proc.stdout.strip().splitlines()[0]
        name, total, used, free = [x.strip() for x in first.split(",")[:4]]
        return {
            "available": True,
            "name": name,
            "total_gb": round(float(total) / 1024.0, 3),
            "used_gb": round(float(used) / 1024.0, 3),
            "free_gb": round(float(free) / 1024.0, 3),
        }
    except Exception as exc:
        return {"available": False, "reason": repr(exc)[:240]}


class HiveTelemetrySampler:
    """Lightweight background GPU sampler so the manifest shows real T4 pressure."""

    def __init__(self, *, enabled: bool = True, interval_seconds: float = 4.0):
        self.enabled = bool(enabled)
        self.interval_seconds = max(1.0, float(interval_seconds or 4.0))
        self.samples: List[Dict[str, Any]] = []
        self._stop = threading.Event()
        self._thread: Optional[threading.Thread] = None

    def _sample_once(self) -> None:
        snap = gpu_memory_snapshot()
        snap["ts"] = time.time()
        self.samples.append(snap)

    def _run(self) -> None:
        while not self._stop.is_set():
            self._sample_once()
            self._stop.wait(self.interval_seconds)

    def start(self) -> None:
        if not self.enabled:
            return
        self._thread = threading.Thread(target=self._run, name="hive-telemetry", daemon=True)
        self._thread.start()

    def stop(self) -> None:
        if not self.enabled:
            return
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=5)
        self._sample_once()

    def summary(self) -> Dict[str, Any]:
        if not self.enabled:
            return {"enabled": False}
        usable = [s for s in self.samples if s.get("available")]
        if not usable:
            return {"enabled": True, "samples": len(self.samples), "available": False, "last": self.samples[-1] if self.samples else None}
        used_values = [float(s.get("used_gb") or 0.0) for s in usable]
        free_values = [float(s.get("free_gb") or 0.0) for s in usable]
        total = float(usable[-1].get("total_gb") or 0.0)
        peak_used = max(used_values) if used_values else 0.0
        return {
            "enabled": True,
            "available": True,
            "samples": len(self.samples),
            "gpu_name": usable[-1].get("name"),
            "total_gb": round(total, 3),
            "peak_used_gb": round(peak_used, 3),
            "min_free_gb": round(min(free_values), 3) if free_values else None,
            "avg_used_gb": round(sum(used_values) / len(used_values), 3) if used_values else None,
            "peak_utilization": round(peak_used / total, 4) if total else None,
            "last": usable[-1],
        }


def resolve_hive_thread_count(requested: int, article_count: int) -> Tuple[int, Dict[str, Any]]:
    requested = _clamp_int(requested, 1, 10, 1)
    article_count = max(1, int(article_count))
    base = min(requested, article_count)
    snap = gpu_memory_snapshot()
    if not AUTO_SCALE_THREADS_TO_T4 or (INFERENCE_BACKEND or "Auto").lower() == "cpu":
        snap["thread_resolution"] = "manual"
        return base, snap
    if not snap.get("available"):
        snap["thread_resolution"] = "gpu-unavailable-manual-cap"
        return base, snap
    free_gb = float(snap.get("free_gb") or 0.0)
    total_gb = float(snap.get("total_gb") or 0.0)
    usable_gb = max(0.1, min(free_gb, total_gb * float(T4_TARGET_VRAM_UTILIZATION)) - float(MIN_FREE_VRAM_GB_AFTER_LOAD))
    per_worker = max(0.25, float(APPROX_VRAM_GB_PER_WORKER))
    gpu_cap = max(1, min(10, int(math.floor(usable_gb / per_worker))))
    resolved = max(1, min(base, gpu_cap))
    snap.update({
        "thread_resolution": "t4-vram-auto",
        "requested_threads": requested,
        "article_count_cap": article_count,
        "estimated_usable_gb": round(usable_gb, 3),
        "approx_vram_gb_per_worker": per_worker,
        "gpu_thread_cap": gpu_cap,
        "resolved_threads": resolved,
    })
    return resolved, snap


def build_planner_prompt(
    topic: str,
    rag_context: str,
    sim: Dict[str, Any],
    ledger: List[Dict[str, Any]],
    intercom_context: str,
    worker_id: str,
    persona: str,
) -> str:
    cites = ", ".join(item["cite"] for item in ledger)
    return f"""
{PLANNER_PROMPT}

Worker: {worker_id}
Persona: {persona}
Topic: {topic}
Brand context: {BRAND_CONTEXT}
Available citations: {cites}
Simulation tag: {sim.get('tag')}

Retrieved surfaces:
{rag_context}

Live hive intercom surface:
{intercom_context}
""".strip()


def make_summa(session: Gemma4E2BSession, rag_context: str, topic: str, index: int, intercom_context: str = "") -> str:
    if not ENABLE_SUMMA_COMPACTION or not rag_context.strip():
        return ""
    should_compact = (index % 2 == 0) or (random.random() < 0.45)
    if not should_compact:
        return ""
    prompt = f"{SUMMA_PROMPT}\n\nTopic: {topic}\n\nSurface chunks:\n{rag_context}\n\nHive notes:\n{intercom_context}"
    return session.chat(prompt, system_text=MASTER_SYSTEM_PROMPT)


def bridge_intercom(
    session: Gemma4E2BSession,
    topic: str,
    plan: Dict[str, Any],
    rag_context: str,
    intercom_context: str,
    worker_id: str,
    persona: str,
    article_index: int,
    intercom: Optional[HiveIntercom],
) -> str:
    rounds = _clamp_int(INTERCOM_ROUNDS, 0, 3, 0)
    if HIVE_MODE.lower() != "mesh" or rounds <= 0:
        return ""
    notes = []
    citations = list(dict.fromkeys(re.findall(r"⟦SFC:[^⟧]+⟧", rag_context)))[:8]
    for round_number in range(1, rounds + 1):
        prompt = HIVE_BRIDGE_PROMPT_TEMPLATE.format(
            worker_id=worker_id,
            worker_persona=persona,
            topic=topic,
            round_number=round_number,
            plan=json.dumps(plan, ensure_ascii=False, indent=2),
            rag_context=rag_context,
            intercom_context=intercom_context,
        )
        note = normalize_markdown(session.chat(prompt, system_text=MASTER_SYSTEM_PROMPT))
        note = sanitize_text(note, max_chars=INTERCOM_PACKET_CHARS)
        if note:
            notes.append(f"[bridge round={round_number}]\n{note}\n[/bridge]")
            if intercom is not None:
                intercom.publish(worker_id, article_index, topic, "bridge", note, citations=citations, priority=0.78)
    return "\n\n".join(notes)


def article_digest(md: str, plan: Dict[str, Any]) -> str:
    headings = [line.strip() for line in md.splitlines() if line.startswith("#")][:8]
    citations = list(dict.fromkeys(re.findall(r"⟦SFC:[^⟧]+⟧", md)))[:10]
    title = headings[0] if headings else str(plan.get("title", "Untitled"))
    return "\n".join([
        f"title: {title}",
        "headings: " + " | ".join(headings[1:]),
        "citations: " + (", ".join(citations) if citations else "none"),
        "angle: " + sanitize_text(plan.get("angle", ""), max_chars=240),
    ])


def local_article_quality(md: str, ledger: List[Dict[str, Any]], plan: Dict[str, Any]) -> Dict[str, Any]:
    citation_tokens = re.findall(r"⟦SFC:[^⟧]+⟧", md or "")
    unique_citations = list(dict.fromkeys(citation_tokens))
    headings = [line.strip() for line in str(md).splitlines() if line.lstrip().startswith("#")]
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", str(md)) if len(p.strip()) > 80]
    paragraph_keys = [stable_hash(re.sub(r"\s+", " ", p.lower()), 16) for p in paragraphs]
    duplicate_paragraphs = len(paragraph_keys) - len(set(paragraph_keys))
    warn_phrases = [str(x).lower() for x in globals().get("QUALITY_WARN_PHRASES", [])]
    lower_md = str(md).lower()
    phrase_hits = [p for p in warn_phrases if p and p in lower_md]
    required = int(globals().get("QUALITY_MIN_CITATIONS", 1) or 0)
    missing_required_citations = max(0, required - len(unique_citations))
    expected = [c for c in plan.get("required_citations", []) if isinstance(c, str)] if isinstance(plan, dict) else []
    expected_missing = [c for c in expected if c not in unique_citations]
    word_count = len(tokens(re.sub(r"⟦[^⟧]+⟧", " ", str(md))))
    penalties = (
        missing_required_citations * 0.18
        + min(0.30, len(expected_missing) * 0.06)
        + min(0.25, duplicate_paragraphs * 0.08)
        + min(0.20, len(phrase_hits) * 0.05)
        + (0.10 if not headings else 0.0)
    )
    score = max(0.0, min(1.0, 1.0 - penalties))
    return {
        "score": round(score, 3),
        "word_count": word_count,
        "heading_count": len(headings),
        "citation_count": len(unique_citations),
        "ledger_citation_count": len(ledger or []),
        "missing_required_citations": missing_required_citations,
        "expected_citations_missing": expected_missing[:10],
        "duplicate_paragraphs": duplicate_paragraphs,
        "warn_phrase_hits": phrase_hits,
    }


def generate_article(
    session: Gemma4E2BSession,
    topic: str,
    index: int,
    *,
    worker_id: str = "solo",
    persona: str = "single local Gemma writer",
    intercom: Optional[HiveIntercom] = None,
) -> Dict[str, Any]:
    retrieval_query = f"{topic}\n{BRAND_CONTEXT}\n{TARGET_READER}\n{persona}"
    rag_context, ledger = RAG_INDEX.citation_context(retrieval_query, top_k=_clamp_int(RAG_TOP_K, 4, 24, 12))
    sim = simulation_packet(topic, rag_context)
    intercom_context = intercom.snapshot(topic, exclude_worker=worker_id) if intercom else "(solo mode)"

    planner_raw = session.chat(
        build_planner_prompt(topic, rag_context, sim, ledger, intercom_context, worker_id, persona),
        system_text=MASTER_SYSTEM_PROMPT,
    )
    plan = extract_json_object(planner_raw) or fallback_plan(topic, [x["cite"] for x in ledger])
    plan_text = json.dumps(plan, ensure_ascii=False, indent=2)

    if intercom is not None:
        intercom.publish(
            worker_id,
            index,
            topic,
            "plan",
            plan_text,
            citations=list(dict.fromkeys(re.findall(r"⟦SFC:[^⟧]+⟧", plan_text))),
            priority=0.82,
        )

    refreshed_intercom = intercom.snapshot(topic, exclude_worker=worker_id) if intercom else intercom_context
    summa_context = make_summa(session, rag_context, topic, index, refreshed_intercom)
    if summa_context:
        summa_id = stable_hash(summa_context, 10)
        summa_context = f"[summa id=Σ-{summa_id}]\n{summa_context}\n[/summa]"

    bridge_context = bridge_intercom(
        session,
        topic,
        plan,
        rag_context,
        refreshed_intercom,
        worker_id,
        persona,
        index,
        intercom,
    )
    draft_intercom = intercom.snapshot(topic, exclude_worker=worker_id) if intercom else refreshed_intercom

    draft_prompt = DRAFT_PROMPT_TEMPLATE.format(
        worker_id=worker_id,
        worker_persona=persona,
        brand_name=BRAND_NAME,
        brand_url=BRAND_URL,
        brand_voice=BRAND_VOICE,
        target_reader=TARGET_READER,
        words=WORDS_PER_ARTICLE,
        style=TEMPERATURE_STYLE,
        appendix="yes" if INCLUDE_TECHNICAL_APPENDIX else "no",
        hive_mode=HIVE_MODE,
        topic=topic,
        brand_context=BRAND_CONTEXT,
        rag_context=rag_context,
        summa_context=summa_context or "(none)",
        intercom_context=draft_intercom,
        bridge_context=bridge_context or "(none)",
        sim_packet=json.dumps(sim, ensure_ascii=False, indent=2),
        plan=plan_text,
    )

    draft = normalize_markdown(session.chat(draft_prompt, system_text=MASTER_SYSTEM_PROMPT))
    if intercom is not None:
        intercom.publish(worker_id, index, topic, "draft", article_digest(draft, plan), citations=[x["cite"] for x in ledger], priority=0.62)

    depth = str(AGENT_DEPTH or "fast").lower()
    if depth in {"deep", "hive-deep", "max"}:
        guard_intercom = intercom.snapshot(topic, exclude_worker=worker_id) if intercom else draft_intercom
        guarded = session.chat(
            FACT_GUARD_PROMPT
            + "\n\nRetrieved surfaces:\n"
            + rag_context
            + "\n\nHive intercom:\n"
            + guard_intercom
            + "\n\nArticle:\n"
            + draft,
            system_text=MASTER_SYSTEM_PROMPT,
        )
        guarded = normalize_markdown(guarded)
        edited = session.chat(
            EDITOR_PROMPT + "\n\nHive intercom:\n" + guard_intercom + "\n\nArticle:\n" + guarded,
            system_text=MASTER_SYSTEM_PROMPT,
        )
        article = normalize_markdown(edited)
    else:
        article = draft

    article = remove_duplicate_paragraphs(article)
    article = add_frontmatter(article, topic, sim, ledger, worker_id=worker_id)
    article += citation_ledger_markdown(ledger)

    quality = local_article_quality(article, ledger, plan) if bool(globals().get("ENABLE_LOCAL_QUALITY_AUDIT", True)) else {}
    final_digest = article_digest(article, plan)
    if intercom is not None:
        if quality:
            intercom.publish(
                worker_id,
                index,
                topic,
                "quality",
                json.dumps(quality, ensure_ascii=False, sort_keys=True),
                citations=[x["cite"] for x in ledger],
                priority=0.76 if quality.get("score", 1.0) < 0.82 else 0.58,
            )
        intercom.publish(worker_id, index, topic, "final", final_digest, citations=[x["cite"] for x in ledger], priority=0.70)

    return {
        "index": index,
        "topic": topic,
        "worker_id": worker_id,
        "worker_persona": persona,
        "plan": plan,
        "article": article,
        "ledger": ledger,
        "simulation": sim,
        "summa": summa_context,
        "bridge": bridge_context,
        "session_calls": session.call_count,
        "session_seconds": round(session.total_seconds, 2),
        "quality": quality,
    }


def run_threaded_hive(selected_topics: List[str]) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
    requested = _clamp_int(GEMMA_THREAD_COUNT, 1, 10, 1)
    resolved, gpu_snap = resolve_hive_thread_count(requested, len(selected_topics))
    intercom = HiveIntercom(max_items=INTERCOM_MEMORY_ITEMS, packet_chars=INTERCOM_PACKET_CHARS)
    intercom.publish(
        "system",
        0,
        "hive contract",
        "risk",
        "Use local surfaces for proof, intercom for editorial memory, and soften any claim that the surfaces do not support.",
        priority=0.95,
    )
    meta = {
        "requested_threads": requested,
        "resolved_threads": resolved,
        "hive_mode": HIVE_MODE,
        "gpu_snapshot": gpu_snap,
        "started_ts": time.time(),
    }

    print(f"Gemma hive requested {requested} worker(s); resolved to {resolved}.")
    print("GPU snapshot:", json.dumps(gpu_snap, ensure_ascii=False))
    telemetry = HiveTelemetrySampler(
        enabled=bool(globals().get("ENABLE_HIVE_TELEMETRY", True)),
        interval_seconds=float(globals().get("HIVE_TELEMETRY_INTERVAL_SECONDS", 4.0) or 4.0),
    )
    telemetry.start()

    if resolved <= 1:
        results = []
        try:
            with Gemma4E2BSession(inference_backend=INFERENCE_BACKEND, worker_id="gemma-01") as session:
                if bool(globals().get("ENABLE_ENGINE_WARMUP", False)):
                    session.chat(str(globals().get("WARMUP_PROMPT", "Return exactly: ready")), system_text=MASTER_SYSTEM_PROMPT, retries=0)
                    intercom.publish("gemma-01", 0, "warmup", "telemetry", "solo worker warmup complete", priority=0.55)
                for idx, topic in enumerate(selected_topics, 1):
                    print(f"\n[{idx}/{len(selected_topics)} gemma-01] {topic}")
                    try:
                        results.append(generate_article(session, topic, idx, worker_id="gemma-01", persona=worker_persona(1), intercom=intercom))
                    except Exception as exc:
                        results.append({"index": idx, "topic": topic, "worker_id": "gemma-01", "error": repr(exc), "traceback": traceback.format_exc()})
                        print("ERROR:", repr(exc))
        finally:
            telemetry.stop()
        meta["intercom"] = intercom.stats()
        meta["telemetry"] = telemetry.summary()
        meta["elapsed_seconds"] = round(time.time() - meta["started_ts"], 2)
        return results, meta

    task_queue: queue.Queue[Tuple[int, str]] = queue.Queue()
    for idx, topic in enumerate(selected_topics, 1):
        task_queue.put((idx, topic))

    results_by_index: Dict[int, Dict[str, Any]] = {}
    results_lock = threading.Lock()
    print_lock = threading.Lock()

    def log(msg: str) -> None:
        with print_lock:
            print(msg, flush=True)

    def worker_loop(worker_number: int) -> None:
        worker_id = f"gemma-{worker_number:02d}"
        persona = worker_persona(worker_number)
        if THREAD_STARTUP_STAGGER_SECONDS:
            time.sleep(max(0.0, float(THREAD_STARTUP_STAGGER_SECONDS)) * (worker_number - 1))
        try:
            with Gemma4E2BSession(
                inference_backend=INFERENCE_BACKEND,
                worker_id=worker_id,
            ) as session:
                log(f"{worker_id} online: {persona}")
                if bool(globals().get("ENABLE_ENGINE_WARMUP", False)):
                    session.chat(str(globals().get("WARMUP_PROMPT", "Return exactly: ready")), system_text=MASTER_SYSTEM_PROMPT, retries=0)
                    intercom.publish(worker_id, 0, "warmup", "telemetry", "worker warmup complete", priority=0.55)
                while True:
                    try:
                        idx, topic = task_queue.get_nowait()
                    except queue.Empty:
                        break
                    log(f"[{idx}/{len(selected_topics)} {worker_id}] {topic}")
                    try:
                        result = generate_article(session, topic, idx, worker_id=worker_id, persona=persona, intercom=intercom)
                    except Exception as exc:
                        result = {"index": idx, "topic": topic, "worker_id": worker_id, "error": repr(exc), "traceback": traceback.format_exc()}
                        log(f"ERROR {worker_id} article {idx}: {repr(exc)}")
                    with results_lock:
                        results_by_index[idx] = result
                    task_queue.task_done()
                log(f"{worker_id} done after {session.call_count} model call(s), {session.total_seconds:.1f}s model time.")
        except Exception as exc:
            log(f"WORKER STARTUP ERROR {worker_id}: {repr(exc)}")
            with results_lock:
                results_by_index[-worker_number] = {"index": -worker_number, "topic": "worker startup", "worker_id": worker_id, "error": repr(exc), "traceback": traceback.format_exc()}

    try:
        globals()["RESOLVED_LLAMA_PARALLEL"] = resolved
        LLAMA_SERVER.ensure_started(parallel=resolved)
        threads = [
            threading.Thread(target=worker_loop, args=(worker_number,), daemon=False)
            for worker_number in range(1, resolved + 1)
        ]
        for t in threads:
            t.start()
        for t in threads:
            t.join()
    finally:
        telemetry.stop()

    while True:
        try:
            idx, topic = task_queue.get_nowait()
        except queue.Empty:
            break
        results_by_index[idx] = {"index": idx, "topic": topic, "worker_id": "unassigned", "error": "No worker completed this task."}

    ordered = [results_by_index[i] for i in sorted(k for k in results_by_index if k > 0)]
    startup_errors = [results_by_index[i] for i in sorted(k for k in results_by_index if k < 0)]
    meta["intercom"] = intercom.stats()
    meta["telemetry"] = telemetry.summary()
    meta["elapsed_seconds"] = round(time.time() - meta["started_ts"], 2)
    if startup_errors:
        meta["startup_errors"] = startup_errors
    return ordered, meta


def write_article_bundle(result: Dict[str, Any], idx: int) -> Dict[str, Any]:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    topic = result["topic"]
    slug = slugify(topic)
    path = OUTPUT_DIR / f"{idx:02d}-{slug}.md"
    path.write_text(result["article"], encoding="utf-8")
    meta = {
        "index": idx,
        "topic": topic,
        "worker_id": result.get("worker_id"),
        "worker_persona": result.get("worker_persona"),
        "path": str(path),
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "citation_count": len(result.get("ledger") or []),
        "simulation": result.get("simulation"),
        "plan": result.get("plan"),
        "session_calls": result.get("session_calls"),
        "session_seconds": result.get("session_seconds"),
        "quality": result.get("quality"),
    }
    meta_path = OUTPUT_DIR / f"{idx:02d}-{slug}.meta.json"
    meta_path.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return meta

print("T4 llama.cpp Gemma hive pipeline ready.")


T4 llama.cpp Gemma hive pipeline ready.


In [ ]:
# Cell 7 — RUN: generate 1–42 QRoadScan blog Markdown files with the 1–10 worker Gemma llama-cpp-python hive

from __future__ import annotations

import json
import shutil
import time
from pathlib import Path

start_time = time.time()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

selected_topics = expand_topics(BLOG_TOPICS, ARTICLE_COUNT)
manifest = {
    "brand": BRAND_NAME,
    "url": BRAND_URL,
    "article_count": len(selected_topics),
    "created_ts": start_time,
    "model_repo": GEMMA_MODEL_REPO_ID,
    "llama_cpp_python_model_path": str(globals().get("MODEL_PATH", "")),
    "context_size": LLAMA_CONTEXT_SIZE,
    "gpu_layers": LLAMA_N_GPU_LAYERS,
    "agent_depth": AGENT_DEPTH,
    "requested_threads": GEMMA_THREAD_COUNT,
    "secret_runtime_audit": dict(globals().get("SECRET_RUNTIME_AUDIT", {})),
    "colab_secret_audit": dict(globals().get("COLAB_SECRET_AUDIT", {})),
    "outputs": [],
}

print(f"Generating {len(selected_topics)} article(s) with Gemma 4 E2B llama-cpp-python CUDA hive...")
print(f"RAG chunks: {len(RAG_INDEX.chunks)}")

hive_results, hive_meta = run_threaded_hive(selected_topics)
manifest["hive"] = hive_meta

for result in hive_results:
    idx = int(result.get("index") or (len(manifest["outputs"]) + 1))
    if "article" in result:
        meta = write_article_bundle(result, idx)
        manifest["outputs"].append(meta)
        print(f"wrote {meta['path']} ({meta['bytes']} bytes, {meta['citation_count']} citations, worker={meta.get('worker_id')})")
    else:
        err = {
            "index": idx,
            "topic": result.get("topic"),
            "worker_id": result.get("worker_id"),
            "error": result.get("error", "unknown error"),
        }
        manifest["outputs"].append(err)
        print("ERROR:", err)

manifest["elapsed_seconds"] = round(time.time() - start_time, 2)
manifest_path = OUTPUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

index_lines = [
    f"# {BRAND_NAME} Gemma 4 E2B llama-cpp-python Hive Blog Batch",
    "",
    f"Generated articles: {len([x for x in manifest['outputs'] if 'path' in x])}",
    f"Requested workers: {GEMMA_THREAD_COUNT}",
    f"Resolved workers: {hive_meta.get('resolved_threads')}",
    "",
]
for item in manifest["outputs"]:
    if "path" in item:
        rel = Path(item["path"]).name
        index_lines.append(f"- [{rel}](./{rel}) — {item.get('topic','')} — worker `{item.get('worker_id')}`")
    else:
        index_lines.append(f"- ERROR article {item.get('index')}: {item.get('error')}")
(OUTPUT_DIR / "index.md").write_text("\n".join(index_lines), encoding="utf-8")

zip_base = str(OUTPUT_DIR)
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("\nDone.")
print("Manifest:", manifest_path)
print("ZIP:", zip_path)
print("Hive stats:", json.dumps(hive_meta, ensure_ascii=False, indent=2)[:4000])
print("Open the Files panel in Colab, or run the next cell for download links.")


Generating 10 article(s) with Gemma 4 E2B llama-cpp-python CUDA hive...
RAG chunks: 2
Gemma hive requested 4 worker(s); resolved to 4.
GPU snapshot: {"available": true, "name": "Tesla T4", "total_gb": 15.0, "used_gb": 0.003, "free_gb": 14.561, "thread_resolution": "t4-vram-auto", "requested_threads": 4, "article_count_cap": 10, "estimated_usable_gb": 11.8, "approx_vram_gb_per_worker": 0.85, "gpu_thread_cap": 10, "resolved_threads": 4}
llama-cpp-python GPU offload support: True
Loading Gemma 4 through llama-cpp-python: /content/qroadscan_gemma4_llamacpp_python_blog_forge/models/gguf/gemma-4-E2B-it-Q8_0.gguf


llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) (0000:00:04.0) - 14807 MiB free
llama_model_loader: loaded meta data with 44 key-value pairs and 601 tensors from /content/qroadscan_gemma4_llamacpp_python_blog_forge/models/gguf/gemma-4-E2B-it-Q8_0.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = gemma4
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 64
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.950000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 1.000000
llama_model_loader: - kv   5:                         general.size_label str              = 4.6B
llama_model_loade

llama-cpp-python model ready. Requested hive parallelism: 4
gemma-01 online: route-safety systems writer: concrete, calm, operational
[1/10 gemma-01] How QRoadScan turns traffic risk signals into calmer route decisions


Using gguf chat template: {%- macro format_parameters(properties, required) -%}
    {%- set standard_keys = ['description', 'type', 'properties', 'required', 'nullable'] -%}
    {%- set ns = namespace(found_first=false) -%}
    {%- for key, value in properties | dictsort -%}
        {%- set add_comma = false -%}
        {%- if key not in standard_keys -%}
            {%- if ns.found_first %},{% endif -%}
            {%- set ns.found_first = true -%}
            {{ key }}:{
            {%- if value['description'] -%}
                description:<|"|>{{ value['description'] }}<|"|>
                {%- set add_comma = true -%}
            {%- endif -%}
            {%- if value['type'] | upper == 'STRING' -%}
                {%- if value['enum'] -%}
                    {%- if add_comma %},{%- else -%} {%- set add_comma = true -%} {% endif -%}
                    enum:{{ format_argument(value['enum']) }}
                {%- endif -%}
            {%- elif value['type'] | upper == 'ARRAY' -%}

gemma-02 online: product narrative strategist: crisp framing, founder-level clarity--- Llama Reply (debug) ---
```json
{
  "title": "From Chaos to Clarity: How QRoadScan Translates Risk Signals into Calm Decisions",
  "subtitle": "Turning raw traffic data into actionable, confidence-scored route intelligence.",
  "reader_promise": "Learn how QRoadScan compresses complex, changing road signals into a single, readable risk pulse, empowering you to make safer route decisions instantly.",
  "angle": "The transition from overwhelming, real-time traffic data to a simple, confidence-based decision framework.",
  "sections": [
    "The Overload: Navigating Live Traffic Signals",
    "The QRoadScan Pulse: One Reading, One Score",
    "Decoding the Risk: Understanding Practical Reasons",
    "Beyond the Alert: Integrating Hazard Awareness into Planning",
    "Building Confidence: The Power of Route Scan Reports",
    "Dashboard Intelligence: Making Calmer Decisions"
  ],
  "required_citations": 

...

llama_perf_context_print:        load time =    1214.01 ms
llama_perf_context_print: prompt eval time =    2568.33 ms /  5609 tokens (    0.46 ms per token,  2183.91 tokens per second)
llama_perf_context_print:        eval time =   14814.10 ms /   999 runs   (   14.83 ms per token,    67.44 tokens per second)
llama_perf_context_print:       total time =   23372.01 ms /  6608 tokens
llama_perf_context_print:    graphs reused =        995
Llama.generate: 554 prefix-match hit, remaining 5479 prompt tokens to eval


--- Llama Reply (debug) ---
# From Chaos to Clarity: How QRoadScan Translates Risk Signals into Calm Decisions

## Turning raw traffic data into actionable, confidence-scored route intelligence.

The dashboard flashes red. Traffic is dense. Instead of paralysis, QRoadScan compresses that changing signal into one readable pulse: a risk reading, a confidence score, and the exact next step.

Fleet safety teams operate in a constant state of overload. Real-time traffic data streams in—a torrent of speed, congestion, and potential incidents. The challenge isn't collecting this information; it’s making sense of the noise. We need a way to cut through that volume, translating complex, changing road signals into a single, readable risk pulse that empowers instant, safer route decisions.

This shift—from overwhelming data to simple clarity—is the core of our approach ⟦SFC:qroadscan-product-positioning-9ff216#c001@72473a31c4⟧.

### The Overload: Navigating Live Traffic Signals

When you are mana

llama_perf_context_print:        load time =    1214.01 ms
llama_perf_context_print: prompt eval time =    2511.84 ms /  5479 tokens (    0.46 ms per token,  2181.27 tokens per second)
llama_perf_context_print:        eval time =   15342.43 ms /  1034 runs   (   14.84 ms per token,    67.39 tokens per second)
llama_perf_context_print:       total time =   23974.43 ms /  6513 tokens
llama_perf_context_print:    graphs reused =       1029


--- Llama Reply (debug) ---
# From Chaos to Clarity: How QRoadScan Translates Risk Signals into Calm Decisions

## Turning raw traffic data into actionable, confidence-scored route intelligence.

The dashboard flashes red. Traffic is dense. Instead of paralysis, QRoadScan compresses that changing signal into one readable pulse: a risk reading, a confidence score, and the exact next step.

Fleet safety teams operate in a constant state of information overload. They are bombarded by real-time traffic data—a relentless stream of inputs that demand immediate processing. The challenge isn't just seeing the data; it’s processing it fast enough to make a sound decision. This is where QRoadScan steps in. It tackles the deluge by compressing complex, changing road signals into a single, readable risk pulse [intercom rank=1 worker=gemma-02 article=2 channel=plan priority=0.82 sig=fe0da139fa9a]. This shifts the focus away from data overload and toward actionable simplicity.

The core concept is t

Llama.generate: 675 prefix-match hit, remaining 5540 prompt tokens to eval


In [ ]:
# Cell 8 — Colab download helpers

from pathlib import Path

try:
    from google.colab import files
    print("Downloading ZIP bundle...")
    files.download(str(Path(str(OUTPUT_DIR) + ".zip")))
except Exception as exc:
    print("Colab download helper unavailable. Files are here:")
    print(OUTPUT_DIR)
    print(Path(str(OUTPUT_DIR) + ".zip"))
    print("Reason:", exc)
